<a href="https://colab.research.google.com/github/BosunL/SEA-VQA/blob/main/English_Gemma_3_4B_SEA_CVQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Visual Question Answering — Gemma 3 4B (English)

---



Evaluating Gemma 3 4B on a custom Kenyan cultural dataset with 50 questions in Ateso across 14 images.

### Install Necessary Libraries

First, we need to install the `transformers` library to access pre-trained VQA models and `Pillow` for image processing.

In [ ]:
# Install transformers and other necessary libraries
!pip install -q transformers accelerate pillow matplotlib requests


### Import Libraries

Import the required modules from `transformers` and `PIL` (Pillow).

In [ ]:
#STEP 1: Install and Import Libraries

import random
import io
import requests
import torch
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from huggingface_hub import login

print("Libraries imported successfully.")


### Authenticate with Hugging Face

Accept the Gemma license at https://huggingface.co/google/gemma-3-4b-it then run this cell.

In [ ]:
# ============================================================
# STEP 2: Authenticate with Hugging Face
# ============================================================
# You need to accept the Gemma license at:
# https://huggingface.co/google/gemma-3-4b-it
# Then paste your HF token below or use the Colab secrets manager

login(token="INSERT TOKEN")


### Load Pre-trained VQA Model and Processor

We'll use the `Google/gemma-3-4b-it` model, which is a powerful VQA model.

In [ ]:
# ============================================================
# STEP 3: Load Model and Processor
# ============================================================

print("Loading Gemma 3 4B model and processor...")

MODEL_ID = "google/gemma-3-4b-it"

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto"
)

print(f"Model loaded on: {model.device}\n")


### Load Custom Dataset (English)

All 50 Ateso questions from the Kenyan culture dataset. Images are fetched from Google Drive by file ID and cached to avoid redundant downloads.


**PSA: Gemma doesn't handle Google Sheets well, so we had to manually write out the questions. This is why we ran multiple sanity checks on the model, making there be more steps in the process.**

In [ ]:
# ============================================================
# STEP 4: Load Custom Kenyan Culture Dataset
# ============================================================
# Images are loaded from Google Drive using the file ID.
# A cache dict avoids re-downloading the same image multiple times.

CUSTOM_DATA = [
    {
        "ID": "custom_001",
        "Category": "Image 1 Questions",
        "file_id": "14pBCQkb9v2VCsBEsE5NWlpMHrWx_FoN-",
        "eng_question": "Which of the baskets are pictured, what are they called in this country, and what is their purpose?",
        "correct_en": "Uteo. Crafted from reeds or palm leaves. It is traditionally found in households and was used to separate the chaff from grain.Its commonly used to hold cereals, in market places, and in homes.",
        "wrong_en_o1": "Mesob baskets. Individual strands of grass or palm are dyed with vibrant colors and utilized for communal serving holding traditional meals",
        "wrong_en_o2": "Bolga baskets. Made from thick, dried elephant grass. They are also very durable because of the tight weaving and are used as market or beach bags, home decor, and for storage and organization.",
        "wrong_en_o3": "Emotokaa. Kanueraitere nakapoloni itomate adakia inyamata luipwaka katoni luitijimete inyamene ayari namakegwelete",
    },
    {
        "ID": "custom_002",
        "Category": "Image 1 Questions",
        "file_id": "14pBCQkb9v2VCsBEsE5NWlpMHrWx_FoN-",
        "eng_question": "What are the yellow jerry cans used for?",
        "correct_en": "Storage of cooking oils, water, and cereals",
        "wrong_en_o1": "Easy transportation of baskets",
        "wrong_en_o2": "They are used to store dough",
        "wrong_en_o3": "Prolong the shelf life of fresh fish and raw meat",
    },
    {
        "ID": "custom_003",
        "Category": "Image 1 Questions",
        "file_id": "14pBCQkb9v2VCsBEsE5NWlpMHrWx_FoN-",
        "eng_question": "What form of transportation is common for small-scale market vendors?",
        "correct_en": "Bicycle. It is highly affordable since it doesn't depend on fuel. It easily carries smaller harvests to the market, allowing vendors to sell their goods and travel light on the way back.",
        "wrong_en_o1": "Tuktuk. It is affordable and allows for easy transport of many items.",
        "wrong_en_o2": "Car. Since it’s the biggest, market vendors often transport mass amounts of food and spices from recent harvest to sale at the market.",
        "wrong_en_o3": "Boda-boda. The two-wheeled motorcycle or bicycle taxi allows for quickest transport to waiting customers and are found in many rural and urban areas.",
    },
    {
        "ID": "custom_004",
        "Category": "Image 1 Questions",
        "file_id": "14pBCQkb9v2VCsBEsE5NWlpMHrWx_FoN-",
        "eng_question": "What is the significance of the sack?",
        "correct_en": "The sack is often used to store cassava and maize. The bags can be made from plastic and a tightly woven outer layer.",
        "wrong_en_o1": "The sack is frequently used to hold rice, which is cooked as a daily meal and mass-produced.",
        "wrong_en_o2": "The sack usually hold dried beans and better preserves nutrients and quality.",
        "wrong_en_o3": "The sack is used for drying wet seeds. It helps prepare the seeds to be planted, leading to better quality crops.",
    },
    {
        "ID": "custom_005",
        "Category": "Image 2 Questions",
        "file_id": "1x8AKBikOaDExx_LhB-vvuvLQZfuYBpeM",
        "eng_question": "The blue container in the top right corner holds many dried, chipped pieces of a particular food. What food is made from this material, and what process was used to get to this point?",
        "correct_en": "Porridge. It is laid on the floor in thin layers on top of a cloth, left to be sundried, and then broken up into smaller pieces.",
        "wrong_en_o1": "Bread. They are battered in flour, dipped in frying oil, and then let rest until they are cooled.",
        "wrong_en_o2": "Chips. They are sliced into thin pieces, baked in an oven at a high heat until all of their moisture is gone, then dipped in water to cool off.",
        "wrong_en_o3": "Fried rice. The uncooked grains are laid in a row, flattened with a roller, and boiled.",
    },
    {
        "ID": "custom_006",
        "Category": "Image 2 Questions",
        "file_id": "1x8AKBikOaDExx_LhB-vvuvLQZfuYBpeM",
        "eng_question": "The preparations and consumption of many of the foods found in markets incorporate spices and seasonings such as cumin, turmeric, coriander, and ginger. This profile of foods reflects which historical trading relationship?",
        "correct_en": "Indian railroad workers and merchants settling in East Africa.",
        "wrong_en_o1": "British citizens introducing Kenyans to baking and roasting.",
        "wrong_en_o2": "Yoruba traders assimilating into Kenyan markets.",
        "wrong_en_o3": "American settlers bringing over ideas of preservation and recycling.",
    },
    {
        "ID": "custom_007",
        "Category": "Image 2 Questions",
        "file_id": "1x8AKBikOaDExx_LhB-vvuvLQZfuYBpeM",
        "eng_question": "Focusing on the tins in the image, what are they locally called?",
        "correct_en": "Gorogoro",
        "wrong_en_o1": "Kipimo",
        "wrong_en_o2": "Uzito",
        "wrong_en_o3": "Nafaka",
    },
    {
        "ID": "custom_008",
        "Category": "Image 2 Questions",
        "file_id": "1x8AKBikOaDExx_LhB-vvuvLQZfuYBpeM",
        "eng_question": "The metal buckets and containers originally held things such as paint, seeds, and cooking fat. Now what are they used for? What does this demonstrate?",
        "correct_en": "Measurement and storage. Demonstrates recycling and repurposing.",
        "wrong_en_o1": "Preservation. Demonstrates innovative refrigeration and sustenance.",
        "wrong_en_o2": "Market display and branding. Demonstrates commerce.",
        "wrong_en_o3": "Making food portable. Demonstrates transportation and mobility.",
    },
    {
        "ID": "custom_009",
        "Category": "Image 3 Questions",
        "file_id": "1LagZEieb5SzUKwQ2jLfgGOXhjchC-_B0",
        "eng_question": "Why is this structure in the water?",
        "correct_en": "This structure used to be a restaurant a few years ago. Now, all that remains is the original steel structure standing in the water. This is in Lake Victoria and due to rising sea levels, flooding has cut more and more into the local land.",
        "wrong_en_o1": "This structure was built because birds kept flying onto the mainland, resting on people’s homes, and disturbing agricultural work.",
        "wrong_en_o2": "This structure is utilized for capturing fish. The net underneath traps seafood. Unfortunately, the design also attracts birds.",
        "wrong_en_o3": "The structure acts as a scarecrow. The birds scares away pests from meddling with important fishing areas for fishermen.",
    },
    {
        "ID": "custom_010",
        "Category": "Image 3 Questions",
        "file_id": "1LagZEieb5SzUKwQ2jLfgGOXhjchC-_B0",
        "eng_question": "Which species of bird is shown by the black-headed bird with the white body in the top center of the image?",
        "correct_en": "African sacred ibis",
        "wrong_en_o1": "Great Egret",
        "wrong_en_o2": "Yellow-billed Stork",
        "wrong_en_o3": "Cormorant",
    },
    {
        "ID": "custom_011",
        "Category": "Image 3 Questions",
        "file_id": "1LagZEieb5SzUKwQ2jLfgGOXhjchC-_B0",
        "eng_question": "Which species of birds are all seen in the image?",
        "correct_en": "African sacred ibis, Great Egret, Yellow-billed Stork, Cormorant",
        "wrong_en_o1": "Grey Crowned Crane, African Jacana, Sooty Gull, Dwarf Bittern",
        "wrong_en_o2": "Little Grebe, Blacksmith Plover, Whiskered Tern, Hadada Ibis",
        "wrong_en_o3": "White-faced Whistling Duck, Red-knobbed Coot, Hamerkop, Striated Heron",
    },
    {
        "ID": "custom_012",
        "Category": "Image 4 Questions",
        "file_id": "1p6k5mfdOOWABJyA65VLM0TuoOPE7wF8I",
        "eng_question": "What is the most common use of the boats shown in the foreground?",
        "correct_en": "Fishing",
        "wrong_en_o1": "Leisure",
        "wrong_en_o2": "Sea travel",
        "wrong_en_o3": "Transporting goods",
    },
    {
        "ID": "custom_013",
        "Category": "Image 4 Questions",
        "file_id": "1p6k5mfdOOWABJyA65VLM0TuoOPE7wF8I",
        "eng_question": "What body of water is being shown in the background?",
        "correct_en": "Lake Victoria",
        "wrong_en_o1": "The Indian Ocean",
        "wrong_en_o2": "The Red Sea",
        "wrong_en_o3": "The Great Lakes",
    },
    {
        "ID": "custom_014",
        "Category": "Image 4 Questions",
        "file_id": "1p6k5mfdOOWABJyA65VLM0TuoOPE7wF8I",
        "eng_question": "The woman in the front left is wearing a traditional Kenyan cloth. What is it called?",
        "correct_en": "Kitenge",
        "wrong_en_o1": "Kente",
        "wrong_en_o2": "Amauti",
        "wrong_en_o3": "Agbada",
    },
    {
        "ID": "custom_015",
        "Category": "Image 4 Questions",
        "file_id": "1p6k5mfdOOWABJyA65VLM0TuoOPE7wF8I",
        "eng_question": "Based on the text that can be found on the boats, where in Kenya is this photo taken?",
        "correct_en": "Dunga Beach",
        "wrong_en_o1": "Hippo Point",
        "wrong_en_o2": "Kit-Mikayi",
        "wrong_en_o3": "Impala Sanctuary",
    },
    {
        "ID": "custom_016",
        "Category": "Image 5 Questions",
        "file_id": "1sd9LqsAAbxvlXVRG7c0TgyO4lTf0dLHT",
        "eng_question": "How has M-PESA primarily transformed financial inclusion in Kenya?",
        "correct_en": "It has revolutionized Kenya’s financial inclusion as it allows people without bank accounts to securely deposit, withdraw, send, and receive money directly from their mobile phones.",
        "wrong_en_o1": "It has made the financial landscape of Kenya more exclusionary, as it is only offered to specific groups of people.",
        "wrong_en_o2": "It has made little to no difference, as most people don’t see a practical use for it.",
        "wrong_en_o3": "It has completely turned Kenya’s financial inclusion upside down, as it has led to mass inflation as a result of more cash being able to flow into and out of businesses.",
    },
    {
        "ID": "custom_017",
        "Category": "Image 5 Questions",
        "file_id": "1sd9LqsAAbxvlXVRG7c0TgyO4lTf0dLHT",
        "eng_question": "What are these local vendor stalls locally called?",
        "correct_en": "Kiosks",
        "wrong_en_o1": "Booths",
        "wrong_en_o2": "Newsstands",
        "wrong_en_o3": "Concessions",
    },
    {
        "ID": "custom_018",
        "Category": "Image 5 Questions",
        "file_id": "1sd9LqsAAbxvlXVRG7c0TgyO4lTf0dLHT",
        "eng_question": "What is the significance of Coca-Cola being branded on the structure and in Kenya as a whole?",
        "correct_en": "Coca-Cola is the continent's largest bottler, with a bottling plant right in Kisumu. Coca-Cola provides stall owners with free or subsidized branded equipment like refrigerators, umbrellas, and painted storefronts. In exchange, the stall owners agree to stock Coca-Cola products and prominently display the logo.",
        "wrong_en_o1": "Coca-Cola has bought out all local markets in Kenya to create a monopoly over the soda market. They are advertising themselves on the structure to subconsciously make citizens want to buy their products more often.",
        "wrong_en_o2": "Coca-Cola sponsors local businesses and has incentives that pay people to openly wear, paint, and display their products. This has heavily stimulated the local markets but serves as a stark reminder that companies are slowly overtaking everyday life in Kenya.",
        "wrong_en_o3": "There is no significance behind the logo being painted; citizens around Kenya show love towards brands they love by painting them on their houses, cars, and businesses.",
    },
    {
        "ID": "custom_019",
        "Category": "Image 5 Questions",
        "file_id": "1sd9LqsAAbxvlXVRG7c0TgyO4lTf0dLHT",
        "eng_question": "What action is most likely to take place at a structure like this?",
        "correct_en": "Buying small produce and food.",
        "wrong_en_o1": "Selling old clothes for cash.",
        "wrong_en_o2": "Submit tax forms and documents.",
        "wrong_en_o3": "Setting up transportation to a place.",
    },
    {
        "ID": "custom_020",
        "Category": "Image 6 Questions",
        "file_id": "1nATb92NkD0KfWqsiOsPCfeo9s-DXZB-m",
        "eng_question": "What rocks are shown in this image, and what is their significance?",
        "correct_en": "Calcium Stones from Kodiaga. These stones are traditionally consumed by pregnant women to supplement calcium, supporting bone health for both mother and child.",
        "wrong_en_o1": "Granite stones used for construction. Because granite is dense and durable, the pictured stones are used for heavy-duty construction and flooring.",
        "wrong_en_o2": "Limestone found from the shallow parts of Lake Victoria. Its softer texture makes it central to construction and agriculture. It is used for treating wastewater and fills in for food and toothpaste.",
        "wrong_en_o3": "Quarry stones from nearby mines. They are shaped because its raw state is often unusable. Frequently used for moisture resistance and underground construction",
    },
    {
        "ID": "custom_021",
        "Category": "Image 6 Questions",
        "file_id": "1nATb92NkD0KfWqsiOsPCfeo9s-DXZB-m",
        "eng_question": "What are the red balls in plastic and how were they prepared for the market?",
        "correct_en": "Groundnuts. They are prepped, soaked with hot water, and stirred and cooked over a heated pan. After this, the skin darkens and peels. They are cooled and they get crispy.",
        "wrong_en_o1": "Kidney beans. They are sorted and soaked. They are boiled for 10 minutes to denature the toxins. Eventually, the heat is reduced to a gentle simmer for under an hour until they are tender.",
        "wrong_en_o2": "Groundnuts. They are prepped and peeled. The water in a pot is seasoned as it is being boiled. When the water is boiled, the groundnuts are poured in and cooked for around three hours. After, they are soaked for a few hours so the salt penetrates the shell. They are drained and sold.",
        "wrong_en_o3": "Macadamia nuts. They are dehusked then the shells are cracked. They are dry roasted on a baking sheet for 10-12 minutes. During this process, they can be seasoned. They are stored and sold.",
    },
    {
        "ID": "custom_022",
        "Category": "Image 6 Questions",
        "file_id": "1nATb92NkD0KfWqsiOsPCfeo9s-DXZB-m",
        "eng_question": "What is in the white bucket?",
        "correct_en": "KSL Tropical Mints",
        "wrong_en_o1": "Glass marbles",
        "wrong_en_o2": "Commercially-produced blue stones",
        "wrong_en_o3": "Tanzanite gems",
    },
    {
        "ID": "custom_023",
        "Category": "Image 7 Questions",
        "file_id": "1jLX5cxvjljTk0PtMOaXFG5m055KSCptC",
        "eng_question": "What attire is being worn in this image?",
        "correct_en": "Owali sisal skirts",
        "wrong_en_o1": "Hula kahiko skirts",
        "wrong_en_o2": "Piupiu skirts",
        "wrong_en_o3": "Liku",
    },
    {
        "ID": "custom_024",
        "Category": "Image 7 Questions",
        "file_id": "1jLX5cxvjljTk0PtMOaXFG5m055KSCptC",
        "eng_question": "What community is this attire representative of?",
        "correct_en": "Luo",
        "wrong_en_o1": "Maasai",
        "wrong_en_o2": "Somali",
        "wrong_en_o3": "Meru",
    },
    {
        "ID": "custom_025",
        "Category": "Image 7 Questions",
        "file_id": "1jLX5cxvjljTk0PtMOaXFG5m055KSCptC",
        "eng_question": "When and for what is this attire traditionally worn?",
        "correct_en": "By women and used for dancing. It is primarily worn during marriage ceremonies, celebrating newborns, and initiations. The loose fibers allow for free movement and for the skirt to sway and flow to the music.",
        "wrong_en_o1": "By children and used for dancing. It is primarily worn at children's birthday parties and celebrates the freedom of youth and celebration of life. Children dance in circles and play the nyatiti and abu",
        "wrong_en_o2": "By women and men and used for rituals. It is primarily worn during sacred moments for religion and coming of age. The community believes its material represents the creations of the Divine, and is thus holy attire.",
        "wrong_en_o3": "By women and used for fashion. It is primarily worn to show off the vibrant colors visible such as bright pink, purple, and teal. The stripping, softening, and dying fibers from the sisal plant make it very beautiful and this fine craftsmanship makes it an attire worn for fashionable and flashy purposes.",
    },
    {
        "ID": "custom_026",
        "Category": "Image 8 Questions",
        "file_id": "1ZuWu-TWfzj_jE7c0Wgs8W3936X1Du-nQ",
        "eng_question": "What are the items held in the black tape or band?",
        "correct_en": "Natural luffa sponges",
        "wrong_en_o1": "White maize",
        "wrong_en_o2": "Dried plantains",
        "wrong_en_o3": "Sea sponges",
    },
    {
        "ID": "custom_027",
        "Category": "Image 8 Questions",
        "file_id": "1ZuWu-TWfzj_jE7c0Wgs8W3936X1Du-nQ",
        "eng_question": "What is the purpose of the items held in the black tape or band?",
        "correct_en": "Body scrubbers. It is soaked in water and then they are perfect for exfoliation in the shower. They are also biodegradable and natural tools for scrubbing dishes and countertops.",
        "wrong_en_o1": "Consumption. It provides great nutrients such as potassium and vitamin B6. Eating them contributes to better gut health and thus acts as a healthy source of food for locals.",
        "wrong_en_o2": "Their natural liquid. They hold natural liquids that are mineral-rich and upon harvest are drained for these liquids. They are set out until all the liquid from the body is collected.",
        "wrong_en_o3": "Construction. They are torn apart and mixed with cement to fill in gaps in walls, floors, and roofs. It’s flexible and strong material makes it perfect for long-lasting structure enhancements and fixes.",
    },
    {
        "ID": "custom_028",
        "Category": "Image 8 Questions",
        "file_id": "1ZuWu-TWfzj_jE7c0Wgs8W3936X1Du-nQ",
        "eng_question": "What is the stack and on the right and what are the items used for?",
        "correct_en": "Raw jute fiber, which is utilized to make fabrics, bags, and mats. It can also be turned into yarn for weaving.",
        "wrong_en_o1": "Dried banana leaves. They have multiple uses, including being ground up for tea. They are also used to wrap up spices and small foods for preservation and and braided into ingata, which is used when carrying heavy things",
        "wrong_en_o2": "Mukeu, which is harvested to make strong, durable natural ropes. It is also used to weave baskets and traditional kiondo bags",
        "wrong_en_o3": "Stacks of locally made rubber bands. helpful for bundling to hold and bind products locally sold together. Also used for crafting, orthodontics, and transport.",
    },
    {
        "ID": "custom_029",
        "Category": "Image 9 Questions",
        "file_id": "1knmPao_ehrnkrqK_nOt5QRU1XAOAYaoL",
        "eng_question": "What type of fish is pictured?",
        "correct_en": "Omena (Silver Cyprinid)",
        "wrong_en_o1": "Tilapia (Ngege)",
        "wrong_en_o2": "Haplochromines (Fulu)",
        "wrong_en_o3": "Lake Magadi Tilapia (Oreochromis alcalicus)",
    },
    {
        "ID": "custom_030",
        "Category": "Image 9 Questions",
        "file_id": "1knmPao_ehrnkrqK_nOt5QRU1XAOAYaoL",
        "eng_question": "What will this fish be used for after being sold?",
        "correct_en": "Food for humans and livestock feed",
        "wrong_en_o1": "Crushed up to be sold in local drinks",
        "wrong_en_o2": "Bait for game",
        "wrong_en_o3": "Sole cure for specific diseases",
    },
    {
        "ID": "custom_031",
        "Category": "Image 9 Questions",
        "file_id": "1knmPao_ehrnkrqK_nOt5QRU1XAOAYaoL",
        "eng_question": "What does the processing method do?",
        "correct_en": "Concentrates nutrients, making them a dense source of essential dietary intake",
        "wrong_en_o1": "Gets rid of all germs and contaminants, lessening the chance of getting a disease due to intake",
        "wrong_en_o2": "Brings out the natural flavors making it more enjoyable for consumption",
        "wrong_en_o3": "Softens it making it easier to cook in dishes",
    },
    {
        "ID": "custom_032",
        "Category": "Image 9 Questions",
        "file_id": "1knmPao_ehrnkrqK_nOt5QRU1XAOAYaoL",
        "eng_question": "What is the woman pictured doing?",
        "correct_en": "Sorting small fish out on a mesh net to be dried in the sun for preservation",
        "wrong_en_o1": "Sorting small fish out on a mesh net to be put in a dish cooked for dinner",
        "wrong_en_o2": "Sorting small fish out on a mesh net to determine which are diseased or not",
        "wrong_en_o3": "Removing bones and skin off of harvested fish to be sold",
    },
    {
        "ID": "custom_033",
        "Category": "Image 10 Questions",
        "file_id": "1zah984l5GJlKEDbi8fF2qo9YSe4tspai",
        "eng_question": "What are these structures made out of?",
        "correct_en": "Framed with wood, sealed with a mixture containing cow dung. The roofs are also framed with timber, tightly packed with dried grass and reeds, and sealed with cow dung to prevent rain and precipitation wear and tear.",
        "wrong_en_o1": "Framed with wood, sealed with cement and wooden bricks. The roofs are also framed with timber, tightly packed with hay and bamboo, and sealed with cement to make it strong.",
        "wrong_en_o2": "The walls are strengthened with gypsum plaster. Corners are coated with lemon eucalyptus oil to repel insects. Small windows and openings for cooling ventilation. The roofs are also framed with timber and tightly packed with dried grass and reeds.",
        "wrong_en_o3": "Framed with sticks and packed with clay for a durable structure. The base in packed with grass and mud to make it comfortable to sleep. Hay is put on top of a roof frame made of bamboo.",
    },
    {
        "ID": "custom_034",
        "Category": "Image 10 Questions",
        "file_id": "1zah984l5GJlKEDbi8fF2qo9YSe4tspai",
        "eng_question": "Who builds these structures in a Luo family?",
        "correct_en": "Both the male and female members of the family split the tasks",
        "wrong_en_o1": "The women in the family",
        "wrong_en_o2": "The eldest son",
        "wrong_en_o3": "The men in the family (the husband and sons)",
    },
    {
        "ID": "custom_035",
        "Category": "Image 10 Questions",
        "file_id": "1zah984l5GJlKEDbi8fF2qo9YSe4tspai",
        "eng_question": "Who lives in the largest hut?",
        "correct_en": "The first wife",
        "wrong_en_o1": "The husband",
        "wrong_en_o2": "The children",
        "wrong_en_o3": "Visiting guests",
    },
    {
        "ID": "custom_036",
        "Category": "Image 10 Questions",
        "file_id": "1zah984l5GJlKEDbi8fF2qo9YSe4tspai",
        "eng_question": "Where was this image likely taken?",
        "correct_en": "Near Kit Mikayi",
        "wrong_en_o1": "Near Maasai Mara",
        "wrong_en_o2": "Near Mt. Kenya",
        "wrong_en_o3": "Near the Kenyan Coastal Region",
    },
    {
        "ID": "custom_037",
        "Category": "Image 10 Questions",
        "file_id": "1zah984l5GJlKEDbi8fF2qo9YSe4tspai",
        "eng_question": "What type of Kenyan hut does this image depict?",
        "correct_en": "Od Mikayi",
        "wrong_en_o1": "The Manyatta / Enkaji",
        "wrong_en_o2": "The Borana / Rendille Domes",
        "wrong_en_o3": "Swahili Coral Huts",
    },
    {
        "ID": "custom_038",
        "Category": "Image 11 Questions",
        "file_id": "15A5HdRDwqnfbXayubjIrqsj5HR6dW0Kb",
        "eng_question": "Who’s clothes are seen on the lines?",
        "correct_en": "A family of multiple ages",
        "wrong_en_o1": "Just baby clothes",
        "wrong_en_o2": "Just women’s clothes",
        "wrong_en_o3": "Just men’s clothes",
    },
    {
        "ID": "custom_039",
        "Category": "Image 11 Questions",
        "file_id": "15A5HdRDwqnfbXayubjIrqsj5HR6dW0Kb",
        "eng_question": "What does this image say about the locals' treatment of clothes?",
        "correct_en": "They handwash it and hang it out on a clothesline",
        "wrong_en_o1": "They use washing and drying machines",
        "wrong_en_o2": "They sew their own clothes",
        "wrong_en_o3": "They collect clothes and often sell it for income",
    },
    {
        "ID": "custom_040",
        "Category": "Image 11 Questions",
        "file_id": "15A5HdRDwqnfbXayubjIrqsj5HR6dW0Kb",
        "eng_question": "What are the animals seen below the clothesline and what is their purpose?",
        "correct_en": "They are the Muscovy duck and are often kept by farmers and backyard poultry keepers. They are quiet birds used for their meat, eggs, and natural pest control.",
        "wrong_en_o1": "They are Khaki Campbell and wander from the nearby Lake Victoria into the villagers backyard to feast on insects in the grass.",
        "wrong_en_o2": "They are chickens and are often used for agricultural purposes. They provide eggs to use in recipes and eat for nutrients, and are often sold or harvested by farmers for their meat.",
        "wrong_en_o3": "They are an invasive duck species that eats surrounding vegetation and insects.",
    },
    {
        "ID": "custom_041",
        "Category": "Image 12 Questions",
        "file_id": "1UlOl57GPNS9cr8MoyA_xCKfbCvNVRPve",
        "eng_question": "What is this man doing?",
        "correct_en": "Hammering steel onto a boat to build it",
        "wrong_en_o1": "Repairing a wooden boat",
        "wrong_en_o2": "Preparing to launch a boat into the water.",
        "wrong_en_o3": "Building a long wooden crate",
    },
    {
        "ID": "custom_042",
        "Category": "Image 12 Questions",
        "file_id": "1UlOl57GPNS9cr8MoyA_xCKfbCvNVRPve",
        "eng_question": "This task will most likely take in total…",
        "correct_en": "A few weeks to a few months",
        "wrong_en_o1": "A day",
        "wrong_en_o2": "A few days",
        "wrong_en_o3": "A few years",
    },
    {
        "ID": "custom_043",
        "Category": "Image 12 Questions",
        "file_id": "1UlOl57GPNS9cr8MoyA_xCKfbCvNVRPve",
        "eng_question": "This man’s job is",
        "correct_en": "A boatmaker",
        "wrong_en_o1": "An engineer",
        "wrong_en_o2": "A fisherman",
        "wrong_en_o3": "A welder",
    },
    {
        "ID": "custom_044",
        "Category": "Image 13 Questions",
        "file_id": "1OpSdDSjVmev0-RXs18Y9_dnmEzJBzFMj",
        "eng_question": "This image shows the most common means of transportation are…",
        "correct_en": "Boda-bodas, tuktuks, cars",
        "wrong_en_o1": "Motorcycles, minibuses, railways",
        "wrong_en_o2": "Bikes and motorcycles",
        "wrong_en_o3": "Cars, motorcycles, ferries",
    },
    {
        "ID": "custom_045",
        "Category": "Image 13 Questions",
        "file_id": "1OpSdDSjVmev0-RXs18Y9_dnmEzJBzFMj",
        "eng_question": "What are the motorcycle taxis in this image called?",
        "correct_en": "Boda-bodas",
        "wrong_en_o1": "Autobikes",
        "wrong_en_o2": "Okada",
        "wrong_en_o3": "Cruisers",
    },
    {
        "ID": "custom_046",
        "Category": "Image 13 Questions",
        "file_id": "1OpSdDSjVmev0-RXs18Y9_dnmEzJBzFMj",
        "eng_question": "Which is a likely popular motorcycle model in this area?",
        "correct_en": "Honda CG125",
        "wrong_en_o1": "Ducati",
        "wrong_en_o2": "BMW Motorrad",
        "wrong_en_o3": "Honda NC750X",
    },
    {
        "ID": "custom_047",
        "Category": "Image 13 Questions",
        "file_id": "1OpSdDSjVmev0-RXs18Y9_dnmEzJBzFMj",
        "eng_question": "Delivery drivers most frequently drive",
        "correct_en": "Motorcycles or bikes",
        "wrong_en_o1": "Cars",
        "wrong_en_o2": "Trucks",
        "wrong_en_o3": "Tuktuks",
    },
    {
        "ID": "custom_048",
        "Category": "Image 14 Questions",
        "file_id": "15fdFEbGyGGQH5565RDtqjSPXCXWcIbyF",
        "eng_question": "This image was taken at a…",
        "correct_en": "Gas station",
        "wrong_en_o1": "Restaurant",
        "wrong_en_o2": "Mechanic stop/auto repair",
        "wrong_en_o3": "Water filling station",
    },
    {
        "ID": "custom_049",
        "Category": "Image 14 Questions",
        "file_id": "15fdFEbGyGGQH5565RDtqjSPXCXWcIbyF",
        "eng_question": "What is in the bottle on the shelf?",
        "correct_en": "Soap / Multipurpose Cleaner",
        "wrong_en_o1": "Laundry detergent",
        "wrong_en_o2": "Soda",
        "wrong_en_o3": "Water",
    },
    {
        "ID": "custom_050",
        "Category": "Image 14 Questions",
        "file_id": "15fdFEbGyGGQH5565RDtqjSPXCXWcIbyF",
        "eng_question": "What is the company featured on the red shirt?",
        "correct_en": "National Oil Corporation of Kenya",
        "wrong_en_o1": "Rubis Energy Kenya",
        "wrong_en_o2": "KFC",
        "wrong_en_o3": "Safaricom",
    },
    {
        "ID": "custom_051",
        "Category": "Image 14 Questions",
        "file_id": "15fdFEbGyGGQH5565RDtqjSPXCXWcIbyF",
        "eng_question": "What is this person’s job?",
        "correct_en": "Dispensing fuel and processing payments",
        "wrong_en_o1": "Serving food to waiting customers",
        "wrong_en_o2": "Driving taxis and being a mechanic",
        "wrong_en_o3": "Cleaning up and being a janitor",
    },
]

def load_drive_image(file_id, _cache={}):
    """Download a Google Drive image by file ID, with caching."""
    if file_id in _cache:
        return _cache[file_id]
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        img = Image.open(io.BytesIO(r.content)).convert("RGB")
        _cache[file_id] = img
        return img
    except Exception as e:
        print(f"  Warning: could not load image {file_id}: {e}")
        return Image.new("RGB", (224, 224), color="gray")

print(f"Custom dataset: {len(CUSTOM_DATA)} questions across 14 images.")


### Preview Dataset

In [ ]:
# ============================================================
# STEP 5: Preview Dataset — First 2 Samples
# ============================================================
print("--- Sample Preview: Custom Kenyan Dataset ---\n")

for i, sample in enumerate(CUSTOM_DATA[:2]):
    print(f"Sample {i+1}")
    print(f"  ID:         {sample['ID']}")
    print(f"  Category:   {sample['Category']}")
    print(f"  Question:   {sample['eng_question']}")
    print(f"  Correct:    {sample['correct_en']}")
    print(f"  Wrong 1:    {sample['wrong_en_o1']}")
    print(f"  Wrong 2:    {sample['wrong_en_o2']}")
    print(f"  Wrong 3:    {sample['wrong_en_o3']}")
    print()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for i, sample in enumerate(CUSTOM_DATA[:2]):
    img = load_drive_image(sample['file_id'])
    axes[i].imshow(img)
    axes[i].set_title(f"Sample {i+1}\n{sample['Category'][:20]}", fontsize=8)
    axes[i].axis("off")
plt.suptitle("Custom Kenyan Dataset — First 2 Samples", fontsize=11)
plt.tight_layout()
plt.show()


### Helper Functions

In [ ]:
# ==========================================================
# STEP 6: Define Helper Functions
# ===========================================================

def build_mcqa_prompt(question, choices):
    """
    Format question + choices into a multiple choice prompt.
    Instructs the model to respond with only A, B, C, or D.
    """
    prompt = (
        "You are answering a visual multiple choice question. "
        "You must respond with ONLY a single letter: A, B, C, or D.\n\n"
        "Example:\n"
        "Question: What color is the sky?\n"
        "A. Red\nB. Blue\nC. Green\nD. Yellow\n"
        "Answer: B\n\n"
        "Now answer this question:\n"
        f"Question: {question}\n"
    )
    for label, choice in zip(["A", "B", "C", "D"], choices):
        prompt += f"{label}. {choice}\n"
    prompt += "Answer:"
    return prompt


def extract_predicted_label(predicted_text, choices):
    """
    Extract predicted label (A/B/C/D) from model output.
    Strategy 1: look for A/B/C/D letter in model output.
    Strategy 2: if no letter found, match answer text directly.
    """
    for label in ["A", "B", "C", "D"]:
        if label in predicted_text.upper():
            return label
    for label, choice in zip(["A", "B", "C", "D"], choices):
        if choice.lower() in predicted_text.lower():
            return label
    return None


### Gemma Inference Function

In [ ]:
# ==========================================================
# STEP 7: Helper Function to Run the Gemma Model
# ===========================================================

def run_gemma(image, prompt, processor, model):
    """
    Run Gemma 3 4B inference on an image + text prompt.
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": prompt}
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device, dtype=torch.bfloat16)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

    input_len = inputs["input_ids"].shape[-1]
    predicted_text = processor.decode(
        outputs[0][input_len:],
        skip_special_tokens=True
    ).strip()

    return predicted_text


### Evaluation Function

In [ ]:
# ============================================================
# STEP 8: Define Evaluation Function
# ============================================================

def evaluate_custom_dataset(data, processor, model):
    correct = 0
    predictions = []
    total = len(data)

    print(f"\nRunning evaluation on {total} samples...")
    print("-" * 40)

    for i, sample in enumerate(data):
        if (i + 1) % 10 == 0 or i == 0:
            print(f"  Processing sample {i+1}/{total}...")

        question    = sample["eng_question"]
        correct_ans = sample["correct_en"]
        choices     = [
            correct_ans,
            sample["wrong_en_o1"],
            sample["wrong_en_o2"],
            sample["wrong_en_o3"]
        ]
        random.shuffle(choices)
        correct_label = ["A", "B", "C", "D"][choices.index(correct_ans)]

        prompt = build_mcqa_prompt(question, choices)

        try:
            image           = load_drive_image(sample["file_id"])
            predicted_text  = run_gemma(image, prompt, processor, model)
            predicted_label = extract_predicted_label(predicted_text, choices)
            is_correct      = (predicted_label == correct_label)
            if is_correct:
                correct += 1
        except Exception as e:
            print(f"  Error on sample {i+1}: {e}")
            predicted_text  = "ERROR"
            predicted_label = None
            is_correct      = False

        predictions.append({
            "sample_id":       sample["ID"],
            "category":        sample["Category"],
            "question":        question,
            "choices":         {l: c for l, c in zip(["A","B","C","D"], choices)},
            "correct_answer":  correct_ans,
            "correct_label":   correct_label,
            "predicted_text":  predicted_text,
            "predicted_label": predicted_label,
            "is_correct":      is_correct
        })

    accuracy = (correct / total) * 100 if total > 0 else 0
    return accuracy, predictions


### Sanity Check (5 Samples)

In [ ]:
# ============================================================
# STEP 9: Sanity Check on First 5 Samples
# ============================================================
print("\nRunning sanity check on 5 samples...")

_, sanity_predictions = evaluate_custom_dataset(
    CUSTOM_DATA[:5], processor, model
)

print("\nSanity check results:")
for i, pred in enumerate(sanity_predictions, 1):
    status = "✓" if pred["is_correct"] else "✗"
    print(f"  [{status}] Q{i}: {pred['question']}")
    print(f"        Predicted: {pred['predicted_label']} "
          f"| Correct: {pred['correct_label']} "
          f"| Model output: {pred['predicted_text']}")

print("\nIf results look reasonable, proceed to full evaluation...\n")


### Full Evaluation (50 Questions)

In [ ]:
# ============================================================
# STEP 10: Run Full Evaluation (50 questions)
# ============================================================
accuracy, all_predictions = evaluate_custom_dataset(
    CUSTOM_DATA, processor, model
)


### Results

In [ ]:
# ============================================================
# STEP 11: Print Final Results
# ============================================================
print("\n" + "="*60)
print(f"  Final Results — Gemma 3 4B | Custom Kenyan Culture Dataset (English)")
print("="*60)

category_scores = {}
for pred in all_predictions:
    cat = pred["category"]
    if cat not in category_scores:
        category_scores[cat] = {"correct": 0, "total": 0}
    category_scores[cat]["total"] += 1
    if pred["is_correct"]:
        category_scores[cat]["correct"] += 1

print("\nPer-image accuracy:")
for cat, scores in sorted(category_scores.items()):
    cat_acc = (scores["correct"] / scores["total"]) * 100
    print(f"  {cat:<45} {cat_acc:.1f}%  ({scores['correct']}/{scores['total']})")

print(f"\nOverall Accuracy:        {accuracy:.2f}%")
print(f"Correct:                 {sum(p['is_correct'] for p in all_predictions)}/{len(all_predictions)}")
print(f"Random chance baseline:  25.00% (4 choices)")
print("="*60)


### Visualise Sample Results

In [ ]:
# ============================================================
# STEP 12: Visualise Results
# ============================================================

import math

# Create a dictionary to group predictions by image file_id
image_performance_map = {}
for i, sample in enumerate(CUSTOM_DATA):
    file_id = sample["file_id"]
    if file_id not in image_performance_map:
        image_performance_map[file_id] = {"image": None, "predictions": []}
    # Load image only once per file_id
    if image_performance_map[file_id]["image"] is None:
        image_performance_map[file_id]["image"] = load_drive_image(file_id)

    # Associate prediction with the correct image based on sample ID
    # Note: all_predictions is 1:1 with CUSTOM_DATA, so we can use index 'i'
    image_performance_map[file_id]["predictions"].append(all_predictions[i])

# Get a sorted list of unique file_ids for consistent plotting order
unique_file_ids = sorted(image_performance_map.keys())
num_unique_images = len(unique_file_ids)

# Determine grid dimensions (e.g., 4 columns)
ncols = 4
nrows = math.ceil(num_unique_images / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
# Flatten the axes array for easier iteration if it's a multi-row grid
axes = axes.flatten() if nrows > 1 or ncols > 1 else [axes] # Handle single subplot case

for i, file_id in enumerate(unique_file_ids):
    img_data = image_performance_map[file_id]
    img = img_data["image"]
    predictions_for_image = img_data["predictions"]

    correct_count = sum(p['is_correct'] for p in predictions_for_image)
    total_questions = len(predictions_for_image)
    image_accuracy = (correct_count / total_questions) * 100 if total_questions > 0 else 0

    # Use the category of the first question associated with this image as a general image label
    image_category = predictions_for_image[0]["category"]

    axes[i].imshow(img)
    color = "green" if image_accuracy == 100 else ("red" if image_accuracy == 0 else "orange")
    axes[i].set_title(
        f"{image_category}\nAcc: {image_accuracy:.1f}% ({correct_count}/{total_questions})",
        fontsize=8, color=color
    )
    axes[i].axis("off")

# Hide any unused subplots
for j in range(num_unique_images, len(axes)):
    if j < len(axes): # Ensure index is valid
        fig.delaxes(axes[j])

plt.suptitle(
    f"Gemma 3 4B — Kenyan Culture Dataset (Ateso) | Overall Accuracy: {accuracy:.1f}% | Random baseline: 25%",
    fontsize=12
)
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to make space for suptitle
plt.show()